# Pre-Worshop Setup
Run this notebook once before the workshop begins. Requires workspace ADMIN.

This notebook:

1. Creates the shared schema and Volumes in the platform-workshop catalog
2. Copies sample docs to the volume
3. Grants participant permissions to users

Prerequisites:

- WORKSPACE ADMIN role
- Databricks CLI installed and authenticated (databricks auth login)

In [0]:
dbutils.widgets.text("catalog", "dennis_schultz", "Catalog Name")
dbutils.widgets.text("schema", "vantage_workshop", "Schema Name")
dbutils.widgets.text("volume_test_documents", "test_documents", "Volume - Test Documents")
dbutils.widgets.text("volume_images", "image_storage", "Volume - Image Storage")

# Unity Catalog
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_test_documents = dbutils.widgets.get("volume_test_documents")
volume_images = dbutils.widgets.get("volume_images")

## Step 1: Create schema and volumes

In [0]:
spark.sql(f"""
  CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}
""")

spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_test_documents}
""")
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_images}
""")

## Step 2: Grant permissions to workshop participants

In [0]:
%sql
GRANT USE CATALOG ON CATALOG ${catalog} TO `account users`;

GRANT USE SCHEMA, SELECT ON SCHEMA ${catalog}.${schema} TO `account users`;

GRANT READ VOLUME ON VOLUME ${catalog}.${schema}.${volume_test_documents}    TO `account users`;
GRANT READ VOLUME ON VOLUME ${catalog}.${schema}.${volume_images} TO `account users`;

SELECT 'Permissions granted' AS status;

## Step 3: Upload sample documents to volume

In [0]:
import shutil
import os

# Derive the bundle root from this notebook's own workspace path.
# Works regardless of which user deployed the bundle or where it was synced.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# e.g. /Users/dennis.schultz/DocumentIntelligenceWorkshop/setup/Pre-workshop Setup
root = "/Workspace" + notebook_path.rsplit("/setup/", 1)[0]

source_dir = f"{root}/SampleDocs"
target_dir = f"/Volumes/{catalog}/{schema}/{volume_test_documents}"

shutil.copytree(
    source_dir,
    target_dir,
    dirs_exist_ok=True
    )

print(f"Copied documents to /Volumes/{catalog}/{schema}/{volume_test_documents}")